# Prática guiada: classificando dígitos do MNIST

Na [primeira parte](1_convolution_ops.ipynb) vimos, na mão, o que uma convolução faz
com uma imagem. Agora deixamos de nos preocupar com os kernels: em vez de escolhê-los, vamos
**treinar uma rede que os aprende sozinha!! uhuuw**.

Pipeline:

1. **Dados**: carregar e preparar o MNIST.
2. **Baseline (rede densa)**: rede neural básica.
3. **CNN**
4. **Data augmentation**: aumentar os dados *dentro* do modelo, durante o treino (*on the fly*).
5. **Visualização**: abrir a rede treinada e olhar os kernels e os feature maps que ela aprendeu *(complementar)*.

## Setup

In [ ]:
"""
Define uma seed global para garantir reprodutibilidade máxima
no treinamento com TensorFlow/Keras.

NOTE 1: Para aulas, deixo essa parte comentada para demonstrar o
        comportamento estocástico das práticas com DL.
NOTE 2: Na prática diária com DL, determinamos as SEEDs somente no 
        split dos dados, pois operações em GPU sofrem perda de 
        performance quando forçadas ao determinismo.
NOTE 3: Em alguns casos de treinamento avançado com DL, não é 
        possível estabelecer 100% de determinismo devido à alta
        dependência de operações paralelizadas.
"""
# 1. Variáveis de ambiente (precisam ser definidas ANTES de importar TF!
import os
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'      # força kernels determinísticos
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'    # determinismo em GPU (cuDNN)


import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, RobustScaler, OneHotEncoder
from sklearn.utils import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from absl import logging

# seeds determinadas após importação dos módulos
keras.utils.set_random_seed(SEED)
# 3. Habilita operações determinísticas no TF (TF >= 2.8)
# NOTE: Haverá perda de performance
# NOTE: Em alguns casos, ainda não será 100% determinístico (paralelismo!)
tf.config.experimental.enable_op_determinism()

# silenciar alguns avisos do tf que não serão relevantes
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)

### Funções auxiliares

In [ ]:
def plot_images(images, titles=None, scale=3):
    """Exibe uma lista de imagens lado a lado"""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(scale * n, scale))
    axes = np.atleast_1d(axes)  # garante iterável quando há uma única imagem

    for i, (ax, img) in enumerate(zip(axes, images)):
        # squeeze remove o eixo de canal: (28, 28, 1) -> (28, 28)
        ax.imshow(np.squeeze(np.asarray(img)), cmap="gray")
        ax.axis("off")
        if titles is not None:
            ax.set_title(titles[i], fontsize=12)

    plt.tight_layout()
    plt.show()


def plot_history(history, skip_epochs=0, smooth=1):
    """Plota as curvas de treino e validação para cada métrica registrada.

    history: o objeto devolvido por model.fit (ou um dict/DataFrame equivalente)
    skip_epochs: ignora as primeiras épocas, cuja escala costuma achatar o resto do gráfico
    smooth: janela da média móvel; 1 desliga a suavização
    """
    df = history.history if hasattr(history, "history") else history
    df = pd.DataFrame(df).iloc[skip_epochs:]

    if smooth > 1:
        janela = np.ones(smooth) / smooth
        df = pd.DataFrame({c: np.convolve(df[c], janela, mode="valid") for c in df.columns})

    # As métricas "base" são as colunas sem o prefixo val_; cada uma vira um subplot
    metricas = [c for c in df.columns if not c.startswith("val_")]

    ncols = 2
    nrows = int(np.ceil(len(metricas) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 4.5), squeeze=False)
    axes = axes.flatten()

    for ax, metrica in zip(axes, metricas):
        ax.plot(df[metrica], linestyle="-", label="treino")
        if f"val_{metrica}" in df.columns:  # nem toda métrica tem par de validação
            ax.plot(df[f"val_{metrica}"], linestyle="--", label="validação")
        ax.set_title(metrica.replace("_", " ").title())
        ax.set_xlabel("Época")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.7)

    # Descarta os subplots que sobraram na última linha
    for ax in axes[len(metricas):]:
        fig.delaxes(ax)

    plt.tight_layout()
    plt.show()

## 1. Dados

O MNIST: 70 mil imagens 28x28 em escala de cinza de dígitos manuscritos,
já divididas em treino (60 mil) e teste (10 mil).

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print(f"Treino: {x_train.shape} imagens, {y_train.shape} rótulos")
print(f" Teste: {x_test.shape} imagens, {y_test.shape} rótulos")
print(f"Faixa de valores dos pixels: {x_train.min()} a {x_train.max()} ({x_train.dtype})")

In [ ]:
# Amostra aleatória do conjunto de treino
idx = np.random.choice(len(x_train), 8, replace=False)
plot_images(x_train[idx], titles=[f"Rótulo: {y}" for y in y_train[idx]])

### Pré-processamento

Duas coisas precisam acontecer antes do treino:

- **Eixo de canal.** As camadas do Keras esperam imagens no formato `(N, altura, largura, canais)`.
  O MNIST vem como `(N, 28, 28)`, então adicionamos o canal (1, por ser escala de cinza).
- **Escala.** Pixels vão de 0 a 255; redes treinam melhor com entradas em torno de 0-1.

O ***rescaling* fica dentro do modelo**, como uma camada
`Rescaling`: assim o modelo recebe a imagem crua e sabe normaliza-la sozinho, e não há risco
de esquecer o passo na hora de fazer uma predição nova.

In [ ]:
# (60000, 28, 28) -> (60000, 28, 28, 1); equivalente ao x_train[..., None]
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32")
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32")

INPUT_SHAPE = x_train.shape[1:]  # (28, 28, 1)

print(f"x_train: {x_train.shape}")
print(f" x_test: {x_test.shape}")
print(f"INPUT_SHAPE: {INPUT_SHAPE}")

### Configuração comum de treino

Os três modelos abaixo são treinados exatamente do mesmo jeito, a única coisa que muda
é a arquitetura. Padronizamos os hiperparâmetros para comparação justa.

In [ ]:
BATCH_SIZE = 1024
EPOCHS = 1000       # teto; quem decide de fato é o EarlyStopping
VALIDATION_SPLIT = 0.2


def make_callbacks(patience=10, checkpoint="checkpoints/temp.keras"):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint,
            monitor="val_loss",
            save_best_only=True,
        )
    ]


def train(model, patience=10, checkpoint="checkpoint/temp.keras", verbose=1):
    """Treina um modelo com a configuração padrão e devolve o history."""
    history = model.fit(
        x_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_split=VALIDATION_SPLIT,
        callbacks=make_callbacks(patience, checkpoint),
        verbose=verbose,
    )
    print("\nTreinamento concluído!")
    return history


def evaluate(model):
    """Avalia no conjunto de teste e devolve a acurácia."""
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"{model.name:>16}  |  loss: {loss:.4f}  |  acurácia: {acc:.4f}")
    return acc

## 2. Baseline: rede densa (FCN)

Antes de usar convolução, vale medir o que se consegue **sem** ela. Vamos utilizar um modelo básico FCN.

In [ ]:
inputs = keras.Input(shape=INPUT_SHAPE)
x = keras.layers.Rescaling(1 / 255.0)(inputs)     # reescalonamento como camada do modelo
x = keras.layers.Flatten()(x)                     # (28, 28, 1) -> (784,)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dense(32, activation="relu")(x)
outputs = keras.layers.Dense(10, activation="softmax")(x)  # 10 classes: dígitos 0-9

model_fcn = keras.Model(inputs, outputs, name="fcn_mnist")

model_fcn.compile(
    optimizer=keras.optimizers.Adam(5e-3),
    # Sparse* porque os rótulos são inteiros (3), e não one-hot ([0,0,0,1,...])
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

model_fcn.summary()

In [ ]:
history_fcn = train(model_fcn, checkpoint="checkpoint/fcn.keras")

In [ ]:
plot_history(history_fcn, skip_epochs=1, smooth=0)

In [ ]:
acc_fcn = evaluate(model_fcn)

## 3. CNN

O bloco que se repete é o da primeira parte da aula: **Conv2D -> ReLU -> MaxPooling**.

- `Conv2D(32, (3, 3))`: 32 kernels 3x3, cada um produzindo um feature map. São esses pesos
  que a rede aprende, no lugar dos kernels que escrevemos à mão na parte 1 *(mais um bias por kernel)*.
- `padding="same"`: mantém a resolução da saída igual à da entrada; quem reduz é o pooling.
- `MaxPooling2D()`: janela 2x2 para mudança de escala piramidal (28x28 -> 14x14 -> 7x7).

Repare no padrão **menos resolução, mais filtros** (32 -> 64): conforme o mapa encolhe,
aumentamos o número de padrões diferentes procurados em cada posição.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)

inputs = keras.Input(shape=INPUT_SHAPE)
x = keras.layers.Rescaling(1 / 255.0)(inputs)

# Bloco 1: 28x28x1 -> 14x14x32
x = keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv1")(x)
x = keras.layers.MaxPooling2D()(x)

# Bloco 2: 14x14x32 -> 7x7x64
x = keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2")(x)
x = keras.layers.MaxPooling2D()(x)

# Achatamento: o mapa 7x7x64 vira um vetor de 3136 features
x = keras.layers.Flatten()(x)

# Camada densa intermediária (OPCIONAL): mais capacidade discriminatória,
# mais risco de overfitting. Experimente!.
# x = keras.layers.Dense(64, activation="relu")(x)
# x = keras.layers.Dropout(0.5)(x) # regularização 

outputs = keras.layers.Dense(10, activation="softmax")(x)

model_cnn = keras.Model(inputs, outputs, name="cnn_mnist")

model_cnn.compile(
    optimizer=keras.optimizers.Adam(5e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

model_cnn.summary()

In [ ]:
history_cnn = train(model_cnn, checkpoint="checkpoint/cnn.keras")

In [ ]:
plot_history(history_cnn, skip_epochs=1, smooth=0)

In [ ]:
acc_cnn = evaluate(model_cnn)

## 4. Data augmentation

E se, a cada época, cada imagem for ligeiramente
diferente? deslocada, girada, zoom... A rede nunca vê duas vezes o mesmo pixel
exato e é estimulada a aprender a *forma* do dígito, não a sua posição.

### DA on-the-fly

No Keras, as camadas de augmentation são camadas como qualquer outra e podem viver na própria
arquitetura. Isso dá três coisas de graça:

- **On the fly**: as transformações acontecem na GPU, batch a batch, sem gerar nem armazenar
  cópias do dataset (offline é *muito pesado*).
- **Só no treino**: essas camadas são ativas apenas quando `training=True` (durante o model.fit()). Em `evaluate()`
  e em `predict()` elas são transparentes/desligadas, deixando a imagem passar intacta.
- **Acompanha o modelo** — ao salvar e recarregar o modelo, o augmentation vai junto,
  e não fica espalhado por um pipeline externo (segurança para reprodutibilidade).

### Escolhendo as transformações com <span style="color:red">cuidado</span>

O augmentation precisa gerar variações **plausíveis para a tarefa**. Em MNIST:

- `RandomTranslation`, `RandomRotation`, `RandomZoom` em doses pequenas, um dígito
  manuscrito de fato poderia aparecer um pouco torto, deslocado ou maior.
- `RandomFlip` seria um **erro**: um 2 espelhado não é um 2, e um 6 de cabeça para baixo
  vira um 9. Aqui o augmentation estaria ensinando rótulos errados!

O `fill_mode="constant"` com `fill_value=0.0` completa as bordas com preto, o mesmo fundo
do MNIST. O padrão (`"reflect"`) espelharia pedaços do traço para dentro da imagem.

In [ ]:
data_augmentation = keras.Sequential(
    [
        keras.layers.RandomTranslation(
            height_factor=0.1, width_factor=0.1,   # até 10% de 28px ≈ 3 pixels
            fill_mode="constant", fill_value=0.0
        ),
        keras.layers.RandomRotation(
            factor=0.06,                           # ±0.06 * 360° ≈ ±20°
            fill_mode="constant", fill_value=0.0
        ),
        keras.layers.RandomZoom(
            height_factor=0.1, width_factor=0.1,
            fill_mode="constant", fill_value=0.0
        ),
    ],
    name="data_augmentation_module",
)

### Vendo o efeito antes de treinar

Antes de colocar o augmentation no modelo, vale **olhar** o que ele produz; é a forma mais
rápida de perceber que os parâmetros estão agressivos demais (dígitos cortados, irreconhecíveis)
ou tímidos demais (saídas idênticas à entrada).

Repare no `training=True`: é ele que liga as transformações. Sem isso, a camada devolveria a
imagem original, que é justamente o comportamento durante a avaliação.

In [ ]:
image = x_train[np.random.choice(x_train.shape[0])]

# A camada espera um batch; repetimos a mesma imagem 6 vezes para ver 6 sorteios diferentes
batch = np.repeat(image[None, ...], 6, axis=0)
augmented = data_augmentation(batch, training=True).numpy()

plot_images([image], titles=["Original"])
plot_images(augmented, titles=[f"Versão {i + 1}" for i in range(len(augmented))])

### A CNN com augmentation

A arquitetura é idêntica à anterior; a única mudança é o *módulo* de augmentation
logo após a entrada, **antes** do `Rescaling`.

Como a rede agora vê um problema mais difícil, ela converge mais devagar, por isso 
subimos a paciência do `EarlyStopping` de 10 para 20 épocas.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)

inputs = keras.Input(shape=INPUT_SHAPE)
x = data_augmentation(inputs)                # ativo apenas durante o treino
x = keras.layers.Rescaling(1 / 255.0)(x)

# Bloco 1: 28x28x1 -> 14x14x32
x = keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv1")(x)
x = keras.layers.MaxPooling2D()(x)

# Bloco 2: 14x14x32 -> 7x7x64
x = keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2")(x)
x = keras.layers.MaxPooling2D()(x)

x = keras.layers.Flatten()(x)
outputs = keras.layers.Dense(10, activation="softmax")(x)

model_aug = keras.Model(inputs, outputs, name="cnn_aug_mnist")

model_aug.compile(
    optimizer=keras.optimizers.Adam(5e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

model_aug.summary()

In [ ]:
# DA exige maior paciência, pois o modelo treina com maior dificuldade
history_aug = train(model_aug, patience=20, checkpoint="checkpoint/cnn_aug.keras")

In [ ]:
plot_history(history_aug, skip_epochs=5, smooth=0)

As curvas agora costumam ter um aspecto diferente: a loss de **treino** fica *pior* que a
das seções anteriores, e pode até ficar acima da de validação. Isso não é um bug — é o
sintoma esperado. O treino passou a ser medido em imagens distorcidas, enquanto a validação
continua sendo feita nas imagens originais. O que interessa é a métrica de validação/teste.

In [ ]:
acc_aug = evaluate(model_aug)

### Comparando os três modelos

In [ ]:
resultados = pd.DataFrame(
    {
        "modelo": ["Densa (FCN)", "CNN", "CNN + augmentation"],
        "acurácia (teste)": [acc_fcn, acc_cnn, acc_aug],
        "parâmetros": [m.count_params() for m in (model_fcn, model_cnn, model_aug)],
    }
)
resultados

In [ ]:
# comparando a proporção dos erros
error_fcn = 1 - acc_fcn
error_cnn = 1 - acc_cnn
error_aug = 1 - acc_aug

fcn_vs_cnn = error_fcn / error_cnn
cnn_vs_aug = error_cnn / error_aug

print(f"erro FCN:CNN = {fcn_vs_cnn:.2f}")
print(f"erro CNN:AUG = {cnn_vs_aug:.2f}")

Observação sobre Augmentation no MNIST:

- O ganho do augmentation no MNIST é relativamente modesto; o dataset é suficientemente grande e limpo, e há pouca
  margm acima de ~99%. O efeito aparece mais forte em datasets pequenos, que é
  exatamente onde a técnica é indispensável.

## 5. BONUS - Abrindo a rede treinada - Seção complementar

Até aqui a CNN foi uma caixa: entra imagem, sai probabilidade. Podemos
olhar dentro dela, em dois níveis:

- **Kernels**: os pesos aprendidos. O que a rede *procura*.
- **Feature maps**: as ativações produzidas por uma imagem específica sob cada kernel aprendido. O que a rede *viu*.

Trabalhamos com o `model_aug`, o modelo final.

### 5.1 Kernels aprendidos

Os pesos de uma `Conv2D` ficam no path do kernel `layer.weights[0]`, num tensor de shape
`(altura, largura, canais_de_entrada, filtros)`. Para a `conv1`: `(3, 3, 1, 32)`, temos
32 kernels 3x3, cada um com um único plano, porque a entrada tem um só canal.

Note a ordem dos eixos: o número de filtros vem **por último**. Para tratar cada kernel como
uma imagem, precisamos trazer esse eixo para a frente com `transpose`.

In [ ]:
conv1 = model_aug.get_layer("conv1")

W, b = conv1.weights
print(f"pesos: {W.shape}   (altura, largura, canais_entrada, filtros)")
print(f"bias:  {b.shape}   (um por filtro)")

In [ ]:
# (3, 3, 1, 32) -> (32, 1, 3, 3): filtros e canais para a frente
kernels = np.transpose(W.numpy(), [3, 2, 0, 1])

# conv1 tem um único canal de entrada, então cada filtro é uma imagem 3x3
kernels_conv1 = kernels[:, 0]

print(f"kernels: {kernels.shape}")
print(f"kernels_conv1: {kernels_conv1.shape}")

In [ ]:
# 8 kernels aleatórios (da camada 1)
n = np.random.choice(kernels_conv1.shape[0], 8, replace=False)
plot_images(kernels_conv1[n], titles=[f"filtro {i}" for i in n], scale=2)

Kernels 3x3 vistos assim dizem pouco a olho nu. Em redes grandes,
treinadas em imagens naturais, os filtros da primeira camada formam figuras reconhecíveis
(bordas em várias orientações, manchas de cor,).
Com 3x3 em dígitos binários, o padrão existe mas é difícil de interpretar diretamente.

> Atenção: só a **primeira** camada é visualizável dessa forma: um filtro da `conv2` tem
> 32 canais empilhados, um para cada feature map de entrada, e não corresponde a nenhuma imagem 
> (é espaço latente hiperdimensional!).

É por isso que, na prática, olhamos para as **ativações** em vez dos pesos.

### 5.2 Remodelando a rede treinada como feature extractor

O truque central desta seção: uma vez treinada, uma CNN pode ser **recortada**. Como um
modelo funcional do Keras é um grafo, podemos declarar um modelo novo que reaproveita a
mesma entrada mas termina numa camada intermediária:

> feature_extractor = keras.Model(model.input, model.get_layer("conv1").output)


Nenhum peso é copiado ou retreinado: o novo modelo **compartilha** os tensores do original.
A saída deixa de ser "a probabilidade de cada dígito" e passa a ser "os 32 feature maps da
primeira camada".

Esse é exatamente o padrão usado em **transfer learning**: pega-se uma rede já treinada,
descarta-se a cabeça de classificação e usa-se o resto como extrator de features
para uma tarefa nova.

In [ ]:
# Os dois tensores que definem o recorte
print("entrada:", model_aug.input.shape)
print("saída da conv1:", model_aug.get_layer("conv1").output.shape)

In [ ]:
def make_feature_extractor(model, layer_name):
    """Recorta um modelo treinado, transformando uma camada interna na saída."""
    return keras.Model(
        inputs=model.input, # mesma entrada do modelo original
        outputs=model.get_layer(layer_name).output, # saída de uma camada específica
        name=f"fe_{layer_name}",
    )


fe_conv1 = make_feature_extractor(model_aug, "conv1")
fe_conv1.summary()

Compare os dois `summary()`: o extractor é literalmente o começo do `model_aug`, truncado.
Repare também que o `data_augmentation` continua lá e é inofensivo, porque chamamos o
modelo em modo de inferência, onde essas camadas deixam a imagem passar intacta.

### 5.3 Feature maps da primeira camada

Passamos uma amostra de imagens pelo extractor e recebemos, para cada uma, 32 mapas 28x28:
um por filtro. Cada mapa mostra **onde na imagem aquele filtro encontrou o padrão que procura**, claro onde a resposta é forte, preto onde a ReLU zerou.

In [ ]:
sample = x_train[:8]
feature_maps = fe_conv1.predict(sample, verbose=0)

print(f"entrada:      {sample.shape}   (imagens, altura, largura, canais)")
print(f"feature maps: {feature_maps.shape}   (imagens, altura, largura, filtros)")

In [ ]:
img_idx = 1 # observar uma imagem específica no batch

# (28, 28, 32) -> (32, 28, 28): canais de features passam a ser "imagens" de feature maps
maps = np.transpose(feature_maps[img_idx], [2, 0, 1])

n = 6
plot_images([sample[img_idx]], titles=[f"Entrada — dígito {y_train[img_idx]}"])
plot_images(maps[:n], titles=[f"feature map {i}" for i in range(n)], scale=2.5)

Compare os mapas entre si: É parecido com o efeito dos kernels desenhados à mão na parte 1,
só que aqui ninguém escolheu esses padrões, eles *emergiram* via *gradient descent*.

Alguns mapas saem quase inteiramente pretos: são filtros que não encontraram nada nesta
imagem em particular (ou filtros pouco usados pela rede). Isso também é informação.

### 5.4 Camadas mais profundas

Trocando o nome da camada, o mesmo código olha mais fundo. A `conv2` vem depois de um
pooling, então seus mapas têm metade da resolução (14x14) e são muito mais abstratos:
em vez de bordas, respondem a combinações de bordas.

A regra geral: **quanto mais fundo, menor a resolução e menos "visual" o mapa**, deixa de
ser algo que se reconhece como o dígito e passa a ser uma assinatura de padrões.

In [ ]:
fe_conv2 = make_feature_extractor(model_aug, "conv2")
feature_maps_2 = fe_conv2.predict(sample, verbose=0)

print(f"conv1: {feature_maps.shape}")
print(f"conv2: {feature_maps_2.shape}")

In [ ]:
img_idx = 1
maps_2 = np.transpose(feature_maps_2[img_idx], [2, 0, 1])

# Uma amostra aleatória entre os 64 filtros da conv2
idx = np.random.choice(len(maps_2), 6, replace=False)

plot_images([sample[img_idx]], titles=[f"Entrada — dígito {y_train[img_idx]}"])
plot_images(maps_2[idx], titles=[f"feature map {i}" for i in idx], scale=2.5)

### 5.5 Um mesmo filtro, várias imagens

As visualizações anteriores fixaram a imagem e variaram o filtro. Fixando o **filtro** e
variando a imagem, a pergunta muda: *o que este filtro procura?*

A resposta aparece na comparação, o mesmo filtro responde à mesma característica em
dígitos diferentes, e isso que significa dizer que a rede aprendeu uma representação generalista.

In [ ]:
sample = x_train[:6]
feature_maps = fe_conv1.predict(sample, verbose=0)

# (imagens, altura, largura, filtros) -> (imagens, filtros, altura, largura)
# (0, 1, 2, 3) -> (0, 3, 1, 2)
maps_por_filtro = np.transpose(feature_maps, [0, 3, 1, 2])

# Sorteia um FILTRO (eixo 1), não uma imagem; são 32 filtros na conv1
filtro = np.random.randint(maps_por_filtro.shape[1])
print(f"Filtro {filtro} de {maps_por_filtro.shape[1]}")

plot_images([kernels_conv1[filtro]], titles=[f"Kernel do filtro {filtro}"], scale=2.5)
plot_images(sample, titles=[f"dígito {y}" for y in y_train[:len(sample)]], scale=2.5)
plot_images(maps_por_filtro[:, filtro], titles=[f"filtro {filtro}"] * len(sample), scale=2.5)

Rode a célula acima algumas vezes: cada execução sorteia um filtro diferente, e dá para ir
identificando quais detectam borda superior, borda lateral, interior do traço, e quais
parecem não fazer nada.

---

### Exercícios propostos

- Aumente as intensidades do `data_augmentation` até os dígitos ficarem irreconhecíveis e
  veja a acurácia cair: augmentation em excesso também atrapalha.
- Treine com apenas 1000 imagens (`x_train[:1000]`) com e sem augmentation. A diferença,
  que aqui foi pequena, deve melhorar.
- Descomente a camada `Dense(64)` da CNN e compare o número de parâmetros e o overfitting.
  Ou experimente outros níveis de MLP após o flatten
- Adicione um terceiro bloco convolucional e visualize a `conv3` na seção 5.4.
- Seja ousada(o)! Quebre tudo :p